# Flower Classification - Day 1: understand and prepare the data

**Goal:** turn the flower photos into audited, reproducible datasets ready for a CNN.
No model is trained in this notebook.

1. In Colab, select a GPU runtime if one is available. Today's data checks also work on CPU.
2. Run the cells in order. The first run downloads a 218 MiB archive.
3. Check the figures and answer the short review questions.
4. Download the Day 1 backup at the end, before closing Colab.

We use five folder labels: daisy, dandelion, roses, sunflowers, and tulips.
The project brief asks you to get your mentor's approval for the dataset.

**Later:** custom CNN -> augmentation experiment -> frozen ResNet18 -> full fine-tuning
-> evaluation and Grad-CAM -> demo and report.

## 1. Check the environment

Colab usually already has PyTorch. Preserve its installed torch/torchvision pair.
Install only missing packages and record the actual versions.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

package_modules = {"Pillow": "PIL", "numpy": "numpy", "matplotlib": "matplotlib",
                   "torch": "torch", "torchvision": "torchvision"}
missing = [package for package, module in package_modules.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
current = Path.cwd()
ROOT = Path("/content/flower-cnn") if IN_COLAB else (current.parent if current.name == "notebooks" else current)
ROOT.mkdir(parents=True, exist_ok=True)
print("Project directory:", ROOT)

## 2. Load the project helpers

The next cell contains the same helper code as the project scripts, so this notebook
works as a standalone upload. Run the cell without editing the embedded source.
Existing local helper files are preserved; Colab receives its own copies.

In [ ]:
# @title Set up the reusable project helpers
HELPER_SOURCES = {'flower_data.py': '"""Day 1: verified download, image audit, and reproducible split manifests.\n\nRun: python flower_data.py --root .\nOnly Pillow is needed. No training or image modification happens here.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport random\nimport shutil\nimport tarfile\nimport tempfile\nimport urllib.request\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nfrom PIL import Image, ImageDraw, ImageOps, UnidentifiedImageError\n\nDATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"\nARCHIVE_SHA256 = "4c54ace7911aaffe13a365c34f650e71dd5bf1be0a58b464e5a7183e3e595d9c"\nARCHIVE_BYTES = 228813984\nCHECKSUM_SOURCE = "https://github.com/tensorflow/datasets/blob/master/tensorflow_datasets/datasets/tf_flowers/checksums.tsv"\nCLASS_NAMES = ("daisy", "dandelion", "roses", "sunflowers", "tulips")\nSPLITS = ("train", "val", "test")\nMANIFEST_FIELDS = ("relative_path", "class_name", "label", "width", "height", "file_sha256", "pixel_sha256", "split")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef download_dataset(raw_dir: Path) -> Path:\n    """Verify the official archive before extracting regular files only."""\n    raw_dir = Path(raw_dir).resolve()\n    raw_dir.mkdir(parents=True, exist_ok=True)\n    dataset_dir = raw_dir / "flower_photos"\n    marker = dataset_dir / ".verified_archive_sha256"\n    if dataset_dir.exists():\n        if marker.is_file() and marker.read_text().strip() == ARCHIVE_SHA256:\n            return dataset_dir\n        raise ValueError(f"Unverified existing directory: {dataset_dir}. Use a new --root.")\n\n    archive = raw_dir / "flower_photos.tgz"\n    if not archive.exists():\n        partial = archive.with_suffix(".tgz.part")\n        received, next_notice = 0, 20 * 1024 * 1024\n        print(f"Downloading {ARCHIVE_BYTES / 1024**2:.1f} MiB of flower images...", flush=True)\n        request = urllib.request.Request(DATASET_URL, headers={"User-Agent": "flower-cnn-day1/1.0"})\n        with urllib.request.urlopen(request, timeout=60) as response, partial.open("wb") as out:\n            for block in iter(lambda: response.read(1024 * 1024), b""):\n                out.write(block)\n                received += len(block)\n                if received >= next_notice:\n                    print(f"  {received / 1024**2:.0f} MiB downloaded", flush=True)\n                    next_notice += 20 * 1024 * 1024\n        if partial.stat().st_size != ARCHIVE_BYTES or sha256_file(partial) != ARCHIVE_SHA256:\n            partial.unlink(missing_ok=True)\n            raise ValueError("Dataset download failed its size/SHA-256 check; rerun to retry.")\n        partial.replace(archive)\n    if archive.stat().st_size != ARCHIVE_BYTES or sha256_file(archive) != ARCHIVE_SHA256:\n        raise ValueError(f"Cached archive checksum mismatch: {archive}. Rename it and retry.")\n\n    # Staging keeps a partially extracted dataset from being mistaken for a complete one.\n    with tempfile.TemporaryDirectory(prefix="extract-", dir=raw_dir) as temp:\n        staging = Path(temp).resolve()\n        with tarfile.open(archive, "r:gz") as source:\n            for member in source:\n                parts = PurePosixPath(member.name).parts\n                if not parts or parts[0] != "flower_photos" or ".." in parts or "\\\\" in member.name:\n                    raise ValueError(f"Unexpected archive path: {member.name}")\n                target = staging.joinpath(*parts).resolve()\n                if not target.is_relative_to(staging):\n                    raise ValueError(f"Archive path escapes staging: {member.name}")\n                if member.isdir():\n                    target.mkdir(parents=True, exist_ok=True)\n                elif member.isfile():\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    with source.extractfile(member) as incoming, target.open("wb") as out:\n                        shutil.copyfileobj(incoming, out)\n                else:\n                    raise ValueError(f"Unsupported archive member: {member.name}")\n        extracted = staging / "flower_photos"\n        if not all((extracted / name).is_dir() for name in CLASS_NAMES):\n            raise ValueError("Archive does not contain all five flower categories.")\n        (extracted / ".verified_archive_sha256").write_text(ARCHIVE_SHA256 + "\\n")\n        extracted.rename(dataset_dir)\n    return dataset_dir\n\n\ndef audit_images(dataset_dir: Path) -> tuple[list[dict], dict]:\n    """Decode every image; keep one copy per exact decoded RGB image.\n\n    Identical images with conflicting class labels are all excluded for review.\n    This detects exact duplicates, not crops or other near duplicates.\n    """\n    candidates, corrupt = [], []\n    dataset_dir = Path(dataset_dir)\n    for label, class_name in enumerate(CLASS_NAMES):\n        class_dir = dataset_dir / class_name\n        if not class_dir.is_dir():\n            raise ValueError(f"Missing class directory: {class_dir}")\n        for path in sorted(class_dir.iterdir()):\n            if not path.is_file() or path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:\n                continue\n            relative = path.relative_to(dataset_dir).as_posix()\n            try:\n                with Image.open(path) as image:\n                    rgb = ImageOps.exif_transpose(image).convert("RGB")\n                    rgb.load()\n                    width, height = rgb.size\n                    pixels = hashlib.sha256(f"{width}x{height}:RGB:".encode() + rgb.tobytes()).hexdigest()\n                candidates.append({"relative_path": relative, "class_name": class_name, "label": label,\n                                   "width": width, "height": height, "file_sha256": sha256_file(path),\n                                   "pixel_sha256": pixels})\n            except (OSError, ValueError, UnidentifiedImageError, Image.DecompressionBombError) as error:\n                corrupt.append({"relative_path": relative, "error": str(error)})\n\n    groups = defaultdict(list)\n    for row in candidates:\n        groups[row["pixel_sha256"]].append(row)\n    clean, duplicates, conflicts = [], [], []\n    for group in groups.values():\n        if len({row["label"] for row in group}) > 1:\n            conflicts.append([row["relative_path"] for row in group])\n            continue\n        clean.append(group[0])\n        duplicates.extend({"kept": group[0]["relative_path"], "excluded": row["relative_path"]} for row in group[1:])\n    clean.sort(key=lambda row: row["relative_path"])\n    if any(not any(row["class_name"] == name for row in clean) for name in CLASS_NAMES):\n        raise ValueError("Every class must contain readable, non-conflicting images.")\n    audit = {"images_scanned": len(candidates) + len(corrupt), "readable_images": len(candidates),\n             "usable_images": len(clean), "corrupt_images": corrupt, "excluded_duplicates": duplicates,\n             "conflicting_label_groups": conflicts, "duplicate_detection": "Exact decoded RGB pixels; near duplicates are not detected."}\n    return clean, audit\n\n\ndef stratified_split(rows: list[dict], seed: int = 42) -> list[dict]:\n    """Assign approximately 70/15/15 percent within each class, deterministically."""\n    grouped = defaultdict(list)\n    for row in rows:\n        grouped[row["class_name"]].append(dict(row))\n    result = []\n    for class_name in sorted(grouped):\n        group = sorted(grouped[class_name], key=lambda row: row["relative_path"])\n        if len(group) < 7:\n            raise ValueError(f"Need at least 7 usable images in {class_name} for three splits.")\n        random.Random(f"{seed}:{class_name}").shuffle(group)\n        n_val = int(len(group) * 0.15 + 0.5)\n        n_test = int(len(group) * 0.15 + 0.5)\n        n_train = len(group) - n_val - n_test\n        for index, row in enumerate(group):\n            row["split"] = "train" if index < n_train else "val" if index < n_train + n_val else "test"\n            result.append(row)\n    return sorted(result, key=lambda row: row["relative_path"])\n\n\ndef validate_splits(rows: list[dict]) -> None:\n    if not rows:\n        raise ValueError("Split manifest cannot be empty.")\n    if len({row["relative_path"] for row in rows}) != len(rows):\n        raise ValueError("A file appears more than once in the split manifest.")\n    if len({row["pixel_sha256"] for row in rows}) != len(rows):\n        raise ValueError("Duplicate decoded images remain in the split manifest.")\n    for row in rows:\n        if row["split"] not in SPLITS:\n            raise ValueError(f"Unknown split: {row[\'split\']}")\n        if row["class_name"] not in CLASS_NAMES or int(row["label"]) != CLASS_NAMES.index(row["class_name"]):\n            raise ValueError("Class name and numeric label do not match.")\n    for name in CLASS_NAMES:\n        if {row["split"] for row in rows if row["class_name"] == name} != set(SPLITS):\n            raise ValueError(f"Class {name} is missing from a split.")\n\n\ndef save_sample_grid(rows: list[dict], dataset_dir: Path, destination: Path, seed: int = 42) -> None:\n    """Only training images are shown; labels are folder labels, not predictions."""\n    tile_width, tile_height = 190, 166\n    canvas = Image.new("RGB", (5 * tile_width, 5 * tile_height + 40), "#f5f7fb")\n    draw = ImageDraw.Draw(canvas)\n    draw.text((12, 12), "Training samples: five images per class", fill="#17253c")\n    for class_index, class_name in enumerate(CLASS_NAMES):\n        group = [row for row in rows if row["class_name"] == class_name and row["split"] == "train"]\n        selection = random.Random(f"{seed}:{class_name}:preview").sample(group, min(5, len(group)))\n        for column, row in enumerate(selection):\n            x, y = column * tile_width + 8, class_index * tile_height + 40\n            with Image.open(dataset_dir / row["relative_path"]) as image:\n                thumbnail = ImageOps.fit(ImageOps.exif_transpose(image).convert("RGB"), (174, 134))\n                canvas.paste(thumbnail, (x, y))\n            draw.text((x, y + 138), class_name, fill="#17253c")\n    canvas.save(destination)\n\n\ndef write_json(path: Path, value) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2) + "\\n", encoding="utf-8")\n\n\ndef record_environment(destination: Path) -> None:\n    packages = {}\n    for name in ("Pillow", "numpy", "matplotlib", "torch", "torchvision"):\n        try:\n            packages[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            packages[name] = None\n    write_json(destination, {"python": platform.python_version(), "platform": platform.platform(), "packages": packages})\n\n\ndef prepare_project(root: Path, seed: int = 42, dataset_dir: Path | None = None) -> dict:\n    root = Path(root).resolve()\n    dataset_dir = Path(dataset_dir).resolve() if dataset_dir else download_dataset(root / "data" / "raw")\n    print("Auditing images and checking exact duplicates...", flush=True)\n    rows, audit = audit_images(dataset_dir)\n    rows = stratified_split(rows, seed)\n    validate_splits(rows)\n    processed, reports = root / "data" / "processed", root / "outputs" / "day1"\n    processed.mkdir(parents=True, exist_ok=True)\n    reports.mkdir(parents=True, exist_ok=True)\n    manifest = processed / "splits.csv"\n    with manifest.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=MANIFEST_FIELDS)\n        writer.writeheader()\n        writer.writerows(rows)\n    class_counts = {name: {split: sum(row["class_name"] == name and row["split"] == split for row in rows)\n                           for split in SPLITS} for name in CLASS_NAMES}\n    summary = {"dataset": "Flower Photos", "dataset_url": DATASET_URL, "source_archive_sha256": ARCHIVE_SHA256 if dataset_dir == root / "data" / "raw" / "flower_photos" else None,\n               "dataset_dir": str(dataset_dir), "seed": seed, "split_ratios": {"train": 0.7, "val": 0.15, "test": 0.15},\n               "class_to_idx": {name: label for label, name in enumerate(CLASS_NAMES)},\n               "class_counts": class_counts, "split_counts": dict(Counter(row["split"] for row in rows)),\n               "width_range": [min(row["width"] for row in rows), max(row["width"] for row in rows)],\n               "height_range": [min(row["height"] for row in rows), max(row["height"] for row in rows)],\n               "manifest_sha256": sha256_file(manifest), "audit": audit}\n    write_json(reports / "data_summary.json", summary)\n    write_json(reports / "audit.json", audit)\n    write_json(processed / "class_to_idx.json", summary["class_to_idx"])\n    record_environment(reports / "environment.json")\n    save_sample_grid(rows, dataset_dir, reports / "sample_grid.jpg", seed)\n    lines = ["# Day 1 data audit", "", f"Usable images: {len(rows)}. Seed: {seed}.", "",\n             "| Class | Train | Validation | Test |", "|---|---:|---:|---:|"]\n    lines += [f"| {name} | {counts[\'train\']} | {counts[\'val\']} | {counts[\'test\']} |" for name, counts in class_counts.items()]\n    lines += ["", f"Unreadable files: {len(audit[\'corrupt_images\'])}.",\n              f"Excluded exact duplicates: {len(audit[\'excluded_duplicates\'])}.",\n              f"Conflicting-label groups excluded: {len(audit[\'conflicting_label_groups\'])}.", "",\n              "Manual label review and near-duplicate review are still needed. The class folders provide the labels.",\n              "Only training images appear in sample_grid.jpg. Reuse splits.csv for every experiment.",\n              "No model has been trained; there are no accuracy results yet."]\n    (reports / "data_report.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    print(json.dumps({"usable_images": len(rows), "split_counts": summary["split_counts"], "report": str(reports / "data_report.md")}, indent=2), flush=True)\n    return summary\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", type=Path, default=Path(__file__).resolve().parent)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--dataset-dir", type=Path, help="Use an already downloaded folder containing the five class directories.")\n    args = parser.parse_args()\n    prepare_project(args.root, args.seed, args.dataset_dir)\n', 'flower_pipeline.py': '"""PyTorch datasets and separate transforms for the saved Day 1 splits."""\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageOps\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision import transforms\n\nfrom flower_data import CLASS_NAMES, sha256_file, validate_splits\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n\nclass FlowerDataset(Dataset):\n    def __init__(self, dataset_dir: Path, rows: list[dict], transform):\n        self.dataset_dir = Path(dataset_dir).resolve()\n        self.rows = rows\n        self.transform = transform\n        for row in rows:\n            image_path = (self.dataset_dir / row["relative_path"]).resolve()\n            if not image_path.is_relative_to(self.dataset_dir) or not image_path.is_file():\n                raise ValueError(f"Missing or invalid image path: {image_path}")\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        with Image.open(self.dataset_dir / row["relative_path"]) as source:\n            image = ImageOps.exif_transpose(source).convert("RGB")\n        return self.transform(image), int(row["label"])\n\n\ndef seed_worker(worker_id):\n    worker_seed = torch.initial_seed() % 2**32\n    np.random.seed(worker_seed)\n    random.seed(worker_seed)\n\n\ndef seed_everything(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef make_loaders(root: Path, batch_size=32, num_workers=0, augment=False, seed=42, dataset_dir=None,\n                 crop_scale=(0.75, 1.0)):\n    """Baseline training is unaugmented; val/test are always deterministic.\n\n    Start a fresh set of loaders with the same seed for each experiment.\n    Fixed seeds improve repeatability; identical GPU results across machines\n    and library versions are not guaranteed.\n    """\n    root = Path(root).resolve()\n    if len(crop_scale) != 2 or not 0 < crop_scale[0] <= crop_scale[1] <= 1:\n        raise ValueError("Crop scale must contain increasing area fractions in (0, 1].")\n    manifest = root / "data" / "processed" / "splits.csv"\n    summary = json.loads((root / "outputs" / "day1" / "data_summary.json").read_text())\n    if sha256_file(manifest) != summary["manifest_sha256"]:\n        raise ValueError("Split manifest changed since preparation. Restore it or rerun data preparation.")\n    with manifest.open(newline="", encoding="utf-8") as stream:\n        rows = list(csv.DictReader(stream))\n    validate_splits(rows)\n    dataset_dir = Path(dataset_dir) if dataset_dir else root / "data" / "raw" / "flower_photos"\n    normalise = [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]\n    evaluation_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), *normalise])\n    train_transform = transforms.Compose([\n        transforms.RandomResizedCrop(224, scale=tuple(crop_scale)),\n        transforms.RandomHorizontalFlip(),\n        transforms.RandomRotation(15),\n        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),\n        *normalise,\n    ]) if augment else evaluation_transform\n    seed_everything(seed)\n    loaders = {}\n    for split in ("train", "val", "test"):\n        dataset = FlowerDataset(dataset_dir, [row for row in rows if row["split"] == split],\n                                train_transform if split == "train" else evaluation_transform)\n        generator = torch.Generator().manual_seed(seed)\n        loaders[split] = DataLoader(dataset, batch_size=batch_size, shuffle=split == "train",\n                                    num_workers=num_workers, pin_memory=torch.cuda.is_available(),\n                                    worker_init_fn=seed_worker, generator=generator, drop_last=False)\n    return loaders, list(CLASS_NAMES)\n\n\ndef save_eda_charts(root: Path):\n    """Standard matplotlib charts of counts and original image dimensions."""\n    root = Path(root)\n    config = root / "outputs" / ".matplotlib"\n    config.mkdir(parents=True, exist_ok=True)\n    os.environ.setdefault("MPLCONFIGDIR", str(config.resolve()))\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    with (root / "data" / "processed" / "splits.csv").open(newline="", encoding="utf-8") as stream:\n        rows = list(csv.DictReader(stream))\n    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")\n    bottom = np.zeros(len(CLASS_NAMES))\n    for split, colour in zip(("train", "val", "test"), ("#2962d9", "#e3a11a", "#249977")):\n        counts = [sum(row["class_name"] == name and row["split"] == split for row in rows) for name in CLASS_NAMES]\n        axes[0].bar(CLASS_NAMES, counts, bottom=bottom, color=colour, label=split)\n        bottom += np.array(counts)\n    axes[0].set(title="Images per class and split", ylabel="Number of images")\n    axes[0].legend()\n    axes[1].hist([int(row["width"]) for row in rows], bins=25, alpha=0.65, label="Width", color="#2962d9")\n    axes[1].hist([int(row["height"]) for row in rows], bins=25, alpha=0.6, label="Height", color="#249977")\n    axes[1].set(title="Original image dimensions", xlabel="Pixels", ylabel="Number of images")\n    axes[1].legend()\n    destination = root / "outputs" / "day1" / "eda_charts.png"\n    fig.savefig(destination, dpi=150)\n    plt.close(fig)\n    return destination\n'}
for name, source in HELPER_SOURCES.items():
    destination = ROOT / name
    if IN_COLAB or not destination.exists():
        destination.write_text(source, encoding='utf-8')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from flower_data import prepare_project, record_environment
from flower_pipeline import make_loaders, save_eda_charts, seed_everything
SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Download, audit, and split

The helper verifies the official archive's SHA-256 checksum, decodes every image,
excludes corrupt files, removes exact duplicates, and flags conflicting duplicate labels.
It leaves the original photos unchanged.

Each flower class is split approximately **70% training / 15% validation / 15% test**.
Stratification keeps each class represented in every split. A fixed seed makes the
split repeatable. All later experiments must reuse `data/processed/splits.csv`.

Exact duplicate checks do not catch all crops, alternate views, or near duplicates.
Folder labels still need visual spot checks.

In [ ]:
summary = prepare_project(ROOT, seed=SEED)
print("\nClass          Train   Val  Test")
for name, counts in summary["class_counts"].items():
    print(f"{name:13} {counts['train']:5} {counts['val']:5} {counts['test']:5}")
audit = summary["audit"]
print("\nUnreadable files:", len(audit["corrupt_images"]))
print("Exact duplicate copies excluded:", len(audit["excluded_duplicates"]))
print("Conflicting-label groups excluded:", len(audit["conflicting_label_groups"]))
print("Width range:", summary["width_range"])
print("Height range:", summary["height_range"])

## 4. Explore the images

The grid shows **training images only**. Look for incorrect labels, distracting
backgrounds, multiple flowers, very small flowers, and unusual lighting.
The charts show class balance and why resizing is necessary.

In [ ]:
from IPython.display import Image as DisplayImage, display

display(DisplayImage(filename=str(ROOT / "outputs/day1/sample_grid.jpg")))
chart_path = save_eda_charts(ROOT)
display(DisplayImage(filename=str(chart_path)))

## 5. Build PyTorch image loaders and check one batch

A Dataset reads one photo and its label; a DataLoader assembles photos into batches.
Images become tensors shaped `[batch, 3, 224, 224]`.

The baseline uses deterministic resizing and cropping. Later, a separate experiment
enables random augmentation **only for training**. Validation and test preprocessing
stay deterministic. ImageNet normalization prepares the pipeline for ResNet18.

In [ ]:
loaders, class_names = make_loaders(ROOT, batch_size=32, num_workers=0, augment=False, seed=SEED)
images, labels = next(iter(loaders["train"]))
assert images.ndim == 4 and tuple(images.shape[1:]) == (3, 224, 224)
assert torch.isfinite(images).all()
assert labels.min().item() >= 0 and labels.max().item() < len(class_names)
print("Image tensor:", tuple(images.shape))
print("Label tensor:", tuple(labels.shape))
print("First five labels:", [class_names[index] for index in labels[:5].tolist()])
print("Batches per split:", {name: len(loader) for name, loader in loaders.items()})
record_environment(ROOT / "outputs/day1/environment.json")
print("Day 1 pipeline is ready. No training has run yet.")

## 6. Your notes

Write your answers in the dictionary in the next cell:

- Which class has the most images? Which has the fewest?
- Do any training examples look mislabeled or ambiguous?
- What backgrounds might the model learn instead of the flower?
- Why do we need separate validation and test sets?
- Why must augmentation happen after the split, only on training images?

Use validation data to choose settings and investigate fixes. Keep the test set
for final comparisons. Do not select improvements using test performance.

In [ ]:
import json

my_observations = {
    "largest_and_smallest_classes": "TODO: read the class counts above",
    "ambiguous_training_examples": "TODO: inspect the sample grid",
    "possible_background_bias": "TODO: describe what you noticed",
    "why_validation_and_test_are_separate": "TODO: explain in your own words",
    "why_augmentation_is_training_only": "TODO: explain in your own words",
}
(ROOT / "outputs/day1/my_observations.json").write_text(
    json.dumps(my_observations, indent=2) + "\n", encoding="utf-8"
)
print("Observations saved. Edit the TODO answers and rerun this cell before backing up.")

## 7. Save your work

Colab's runtime storage is temporary. This cell creates a small backup containing
helper scripts, the split manifests, audit reports, figures, exact package versions,
and your notes. It excludes the image archive and photographs; those can be downloaded again.

Save a copy of this notebook to Drive as well. Re-running it recreates the dataset
and the same split when the source data and seed are unchanged.

In [ ]:
import zipfile

exact_versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(ROOT / "outputs/day1/requirements-runtime.txt").write_text(exact_versions, encoding="utf-8")
notes = ROOT / "outputs/day1/my_notes.md"
if not notes.exists():
    notes.write_text("# My Day 1 observations\n\nWrite your observations here.\n", encoding="utf-8")
backup = ROOT / "day1_backup.zip"
with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as archive:
    for relative in ("data/processed", "outputs/day1"):
        for path in sorted((ROOT / relative).rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(ROOT))
    for name in HELPER_SOURCES:
        archive.write(ROOT / name, name)
print("Backup created:", backup)
if IN_COLAB:
    from google.colab import files
    files.download(str(backup))

## Next: the custom CNN

Day 2 will add three convolution blocks with batch normalization and pooling,
a training/validation loop, best-checkpoint saving, and a comparison of training
with and without augmentation. Keep today's split fixed.

Sources:
- [Official Flower Photos dataset](https://www.tensorflow.org/tutorials/load_data/images)
- [Dataset checksum](https://github.com/tensorflow/datasets/blob/master/tensorflow_datasets/datasets/tf_flowers/checksums.tsv)
- [PyTorch Dataset and DataLoader tutorial](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)

Image credits and individual CC-BY attribution are in the downloaded `flower_photos/LICENSE.txt`.